# FPGA ML Accelerator Deployment — PYNQ Z2

This notebook deploys all 5 ML models (KNN, DT, RF, SVM, MLP) on the PYNQ Z2 FPGA
and benchmarks them across 4 datasets (Iris, Wine, Cancer, MNIST).

**Directory layout on PYNQ:**
```
pynq_deploy/
    overlays/           <model>_accel.bit + .hwh
    models/<dataset>/   <model>_params.npz
    test_vectors/<dataset>/  test_data.npz
    deploy_notebook.ipynb    (this file)
```

In [ ]:
import time
import struct
import numpy as np
from pynq import Overlay, allocate

# AXI-Lite offsets
AP_CTRL  = 0x00
AP_START = 0x01
AP_DONE  = 0x02
AP_IDLE  = 0x04

REG_MAP = {
    "knn": {"mode":0x10, "num_features":0x18, "num_classes":0x20,
            "num_train":0x28, "num_test":0x30, "latency_out":0x38},
    "dt":  {"mode":0x10, "num_features":0x18, "num_classes":0x20,
            "num_nodes":0x28, "num_test":0x30, "latency_out":0x38},
    "rf":  {"mode":0x10, "num_features":0x18, "num_classes":0x20,
            "num_trees":0x28, "num_test":0x30, "latency_out":0x38},
    "svm": {"mode":0x10, "num_features":0x18, "num_classes":0x20,
            "n_sv":0x28, "num_test":0x30, "latency_out":0x38},
    "mlp": {"mode":0x10, "num_features":0x18, "num_classes":0x20,
            "h1_size":0x28, "h2_size":0x30, "num_test":0x38,
            "latency_out":0x40},
}

FRAC_BITS = 8
SCALE = 1 << FRAC_BITS

def to_fixed(val):
    v = int(round(val * SCALE))
    return np.int32(max(-32768, min(32767, v)))

def to_u32(v):
    return np.uint32(struct.unpack('<I', struct.pack('<i', int(v)))[0])

def write_reg(ip, off, val):
    ip.write(off, int(val))

def read_reg(ip, off):
    return ip.read(off)

print("Utilities loaded.")

## Model Parameter Buffer Builders

In [ ]:
def build_load_buffer(model_name, params):
    """Build DMA buffer for mode=0 (model loading)."""
    words = []
    
    if model_name == "knn":
        td = params["train_data"]
        tl = params["train_labels"]
        n_train, n_feat = td.shape
        for i in range(n_train):
            for j in range(n_feat):
                words.append(to_u32(td[i,j]))
            words.append(to_u32(tl[i]))
    
    elif model_name == "dt":
        feat = params["feature"].astype(np.int32)
        thresh = params["threshold"].astype(np.int32)
        left = params["children_left"].astype(np.int32)
        right = params["children_right"].astype(np.int32)
        cls = params["node_class"].astype(np.int32)
        for i in range(len(feat)):
            words.extend([to_u32(feat[i]), to_u32(thresh[i]),
                         to_u32(left[i]), to_u32(right[i]), to_u32(cls[i])])
    
    elif model_name == "rf":
        n_trees = int(params["n_trees"])
        tree_n_nodes = params["tree_n_nodes"].astype(np.int32)
        tree_offsets = params["tree_offsets"].astype(np.int32)
        feat = params["feature"].astype(np.int32)
        thresh = params["threshold"].astype(np.int32)
        left = params["children_left"].astype(np.int32)
        right = params["children_right"].astype(np.int32)
        cls = params["node_class"].astype(np.int32)
        for t in range(n_trees):
            nn = tree_n_nodes[t]
            words.append(to_u32(nn))
            off = tree_offsets[t]
            for i in range(nn):
                idx = off + i
                words.extend([to_u32(feat[idx]), to_u32(thresh[idx]),
                             to_u32(left[idx]), to_u32(right[idx]), to_u32(cls[idx])])
    
    elif model_name == "svm":
        gamma = params["gamma"]
        n_support = params["n_support"].astype(np.int32)
        intercept = params["intercept"].astype(np.int32)
        svs = params["support_vectors"].astype(np.int32)
        dc = params["dual_coef"].astype(np.int32)
        n_classes = int(params["n_classes"])
        words.append(to_u32(to_fixed(float(gamma))))
        for c in range(n_classes):
            words.append(to_u32(n_support[c]))
        for v in intercept.flat:
            words.append(to_u32(v))
        for v in svs.flat:
            words.append(to_u32(v))
        for v in dc.flat:
            words.append(to_u32(v))
    
    elif model_name == "mlp":
        for layer_idx in range(3):
            w = params[f"w{layer_idx}"].astype(np.int32)
            b = params[f"b{layer_idx}"].astype(np.int32)
            for v in w.flat:
                words.append(to_u32(v))
            for v in b.flat:
                words.append(to_u32(v))
    
    return np.array(words, dtype=np.uint32)

print("Buffer builders loaded.")

## Core Run Function

In [ ]:
def run_model(model_name, dataset_name, base_dir="."):
    accel = f"{model_name}_accel"
    bit_path = f"{base_dir}/overlays/{accel}.bit"
    
    print(f"\n{'='*60}")
    print(f"  {model_name.upper()} on {dataset_name.upper()}")
    print(f"{'='*60}")
    
    # Load overlay
    ol = Overlay(bit_path)
    ip = getattr(ol, f"{accel}_0")
    dma = ol.axi_dma_0
    regs = REG_MAP[model_name]
    
    # Load data
    params = np.load(f"{base_dir}/models/{dataset_name}/{model_name}_params.npz", allow_pickle=True)
    test = np.load(f"{base_dir}/test_vectors/{dataset_name}/test_data.npz", allow_pickle=True)
    
    n_feat = int(params.get("n_features", test["n_features"]))
    n_cls  = int(params.get("n_classes", test["n_classes"]))
    n_test = int(test["n_test"])
    y_true = test["y_test"][:n_test]
    X_test = test["X_test_sc_q"][:n_test] if model_name in ("svm","mlp") else test["X_test_gs_q"][:n_test]
    
    # === MODE 0: Load model ===
    write_reg(ip, regs["mode"], 0)
    write_reg(ip, regs["num_features"], n_feat)
    write_reg(ip, regs["num_classes"], n_cls)
    
    if model_name == "knn":
        write_reg(ip, regs["num_train"], int(params["n_train"]))
    elif model_name == "dt":
        write_reg(ip, regs["num_nodes"], int(params["n_nodes"]))
    elif model_name == "rf":
        write_reg(ip, regs["num_trees"], int(params["n_trees"]))
    elif model_name == "svm":
        write_reg(ip, regs["n_sv"], int(params["n_sv"]))
    elif model_name == "mlp":
        ls = params["layer_sizes"]
        write_reg(ip, regs["h1_size"], int(ls[1]))
        write_reg(ip, regs["h2_size"], int(ls[2]))
    
    write_reg(ip, regs["num_test"], 0)
    
    load_buf = build_load_buffer(model_name, params)
    in_buf = allocate(shape=(len(load_buf),), dtype=np.uint32)
    in_buf[:] = load_buf
    in_buf.flush()
    
    write_reg(ip, AP_CTRL, AP_START)
    dma.sendchannel.transfer(in_buf)
    dma.sendchannel.wait()
    while not (read_reg(ip, AP_CTRL) & AP_DONE): pass
    
    print(f"  Model loaded ({len(load_buf)} words)")
    in_buf.freebuffer()
    
    # === MODE 1: Inference ===
    write_reg(ip, regs["mode"], 1)
    write_reg(ip, regs["num_test"], n_test)
    
    inf_words = []
    for i in range(n_test):
        for j in range(n_feat):
            inf_words.append(to_u32(X_test[i,j]))
    inf_arr = np.array(inf_words, dtype=np.uint32)
    
    in_buf = allocate(shape=(len(inf_arr),), dtype=np.uint32)
    out_buf = allocate(shape=(n_test,), dtype=np.uint32)
    in_buf[:] = inf_arr
    in_buf.flush()
    
    t0 = time.perf_counter()
    write_reg(ip, AP_CTRL, AP_START)
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    while not (read_reg(ip, AP_CTRL) & AP_DONE): pass
    t1 = time.perf_counter()
    
    out_buf.invalidate()
    preds = np.frombuffer(out_buf, dtype=np.int32)[:n_test].copy()
    hw_cyc = read_reg(ip, regs["latency_out"])
    wall_us = (t1 - t0) * 1e6
    acc = np.sum(preds == y_true) / n_test * 100.0
    
    in_buf.freebuffer(); out_buf.freebuffer()
    
    print(f"  Accuracy:  {acc:.2f}%")
    print(f"  HW Cycles: {hw_cyc}")
    print(f"  Wall Time: {wall_us:.1f} µs")
    
    return {"model": model_name, "dataset": dataset_name,
            "accuracy": acc, "hw_cycles": hw_cyc, "wall_us": wall_us}

print("Run function loaded.")

## Run All 20 Combinations

In [ ]:
MODELS   = ["knn", "dt", "rf", "svm", "mlp"]
DATASETS = ["iris", "wine", "cancer", "mnist"]

results = []
for m in MODELS:
    for d in DATASETS:
        try:
            r = run_model(m, d)
            results.append(r)
        except Exception as e:
            print(f"  FAILED {m}/{d}: {e}")
            results.append({"model": m, "dataset": d,
                           "accuracy": 0, "hw_cycles": 0, "wall_us": 0, "error": str(e)})

## Results Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print("\n" + "="*70)
print("  FPGA DEPLOYMENT RESULTS")
print("="*70)

# Accuracy table
print("\n--- Accuracy (%) ---")
acc_pivot = df.pivot(index="model", columns="dataset", values="accuracy")
acc_pivot = acc_pivot.reindex(index=MODELS, columns=DATASETS)
print(acc_pivot.to_string(float_format="%.2f"))

# HW cycles table
print("\n--- HW Cycles ---")
cyc_pivot = df.pivot(index="model", columns="dataset", values="hw_cycles")
cyc_pivot = cyc_pivot.reindex(index=MODELS, columns=DATASETS)
print(cyc_pivot.to_string())

# Wall time table
print("\n--- Wall Time (µs) ---")
wall_pivot = df.pivot(index="model", columns="dataset", values="wall_us")
wall_pivot = wall_pivot.reindex(index=MODELS, columns=DATASETS)
print(wall_pivot.to_string(float_format="%.1f"))

# Save
df.to_csv("fpga_results.csv", index=False)
print("\nSaved to fpga_results.csv")

## FPGA vs ESP32 Comparison

In [ ]:
# ESP32 latency data (µs) — fill in from ESP32 benchmarks
# Format: esp32_latency[model][dataset] = latency_us
esp32_latency = {
    "knn":  {"iris": None, "wine": None, "cancer": None, "mnist": None},
    "dt":   {"iris": None, "wine": None, "cancer": None, "mnist": None},
    "rf":   {"iris": None, "wine": None, "cancer": None, "mnist": None},
    "svm":  {"iris": None, "wine": None, "cancer": None, "mnist": None},
    "mlp":  {"iris": None, "wine": None, "cancer": None, "mnist": None},
}

# FPGA latency from results (100 MHz clock → 1 cycle = 10 ns)
FPGA_CLK_MHZ = 100.0

print(f"{'Model':<8} {'Dataset':<8} {'FPGA(µs)':>10} {'ESP32(µs)':>10} {'Speedup':>10}")
print("-" * 50)
for r in results:
    m, d = r["model"], r["dataset"]
    fpga_us = r["hw_cycles"] / FPGA_CLK_MHZ  # cycles / MHz = µs
    esp_us = esp32_latency.get(m, {}).get(d)
    if esp_us:
        speedup = f"{esp_us / fpga_us:.1f}x"
    else:
        speedup = "X"
    print(f"{m:<8} {d:<8} {fpga_us:>10.2f} {str(esp_us or 'X'):>10} {speedup:>10}")